# LLM Token Sampling — PMF, Temperature, and Nucleus Sampling

When a language model generates the next token, it outputs a **logit** for every word in its vocabulary. These logits are converted to a probability distribution via the **softmax** function:

$$
P(X = k) = \frac{e^{z_k / T}}{\sum_j e^{z_j / T}}
$$

where $z_k$ is the logit for token $k$ and $T$ is the **temperature**.

This output **is a PMF** — it assigns a probability to each possible next token.

Questions:

1. What does the token PMF look like?
2. How does temperature $T$ reshape the PMF?
3. How does **top-$k$ sampling** truncate the PMF?
4. How does **nucleus (top-$p$) sampling** use the CDF to pick a cutoff?
5. What does **entropy** $H(X) = -E[\log P(X)]$ tell us about model uncertainty?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Simulate a small vocabulary (in practice this is 50k+ tokens)
vocab = [
    "the", "a", "cat", "dog", "sat", "mat", "on", "under",
    "big", "small", "ran", "jumped", "over", "fence", "house", "tree"
]
V = len(vocab)

# Simulate raw logits from a language model
logits = rng.normal(0, 2, size=V)
logits[2] += 3   # make "cat" the most likely token in this context
logits[7] += 1.5 # "under" also likely

## Step 1: Softmax converts logits to a PMF

The softmax output assigns a valid probability to every token: each value is in $[0, 1]$ and they sum to 1.

In [ ]:
def softmax(logits, T=1.0):
    z = logits / T
    z -= z.max()  # numerical stability
    exp_z = np.exp(z)
    return exp_z / exp_z.sum()

pmf = softmax(logits)
order = np.argsort(pmf)[::-1]

plt.bar(range(V), pmf[order])
plt.xticks(range(V), [vocab[i] for i in order], rotation=45, ha="right")
plt.ylabel("P(next token = k)")
plt.title("Token PMF — softmax output")
plt.tight_layout()
plt.show()

print(f"Sum of probabilities: {pmf.sum():.6f}")
print(f"Most likely token: '{vocab[order[0]]}' with P = {pmf[order[0]]:.3f}")

## Step 2: Temperature reshapes the PMF

- $T < 1$ — **sharpens** the distribution: the model becomes more confident, top tokens dominate.
- $T = 1$ — the raw softmax output.
- $T > 1$ — **flattens** the distribution: the model becomes more uniform, more tokens get sampled.

In [ ]:
temperatures = [0.3, 1.0, 2.0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, T in zip(axes, temperatures):
    p = softmax(logits, T=T)
    p_ordered = p[order]
    ax.bar(range(V), p_ordered)
    ax.set_xticks(range(V))
    ax.set_xticklabels([vocab[i] for i in order], rotation=45, ha="right", fontsize=8)
    ax.set_title(f"Temperature T = {T}")
    ax.set_ylabel("Probability")

plt.tight_layout()
plt.show()

## Step 3: Top-$k$ sampling — truncate the PMF

Instead of sampling from the full vocabulary, keep only the $k$ most probable tokens and renormalise:

$$
P_{\text{top-}k}(X = i) = \frac{P(X = i)}{\sum_{j \in \text{top-}k} P(X = j)}
$$

This prevents the model from occasionally sampling very unlikely tokens.

In [ ]:
def top_k_pmf(pmf, k):
    top_k_idx = np.argsort(pmf)[::-1][:k]
    pmf_k = np.zeros_like(pmf)
    pmf_k[top_k_idx] = pmf[top_k_idx]
    pmf_k /= pmf_k.sum()
    return pmf_k

k = 5
pmf_topk = top_k_pmf(pmf, k)

plt.bar(range(V), pmf[order], alpha=0.4, label="Full PMF")
plt.bar(range(V), pmf_topk[order], alpha=0.8, label=f"Top-{k} PMF")
plt.xticks(range(V), [vocab[i] for i in order], rotation=45, ha="right")
plt.ylabel("Probability")
plt.title(f"Top-{k} sampling truncates the PMF")
plt.legend()
plt.tight_layout()
plt.show()

## Step 4: Nucleus (top-$p$) sampling — use the CDF as a cutoff

Top-$p$ sampling keeps the **smallest set of tokens whose cumulative probability exceeds $p$**.

This is a direct application of the CDF:

$$
\text{Keep token } k \text{ if } F(k) = \sum_{j \le k} P(X = j) \le p
$$

where tokens are sorted by probability (highest first). The threshold adapts: when the model is confident, fewer tokens are kept; when it is uncertain, more are kept.

In [ ]:
def nucleus_pmf(pmf, p):
    sorted_idx = np.argsort(pmf)[::-1]
    sorted_probs = pmf[sorted_idx]
    cumulative = np.cumsum(sorted_probs)
    # Keep tokens until cumulative probability exceeds p
    cutoff = np.searchsorted(cumulative, p) + 1
    nucleus_idx = sorted_idx[:cutoff]
    pmf_nucleus = np.zeros_like(pmf)
    pmf_nucleus[nucleus_idx] = pmf[nucleus_idx]
    pmf_nucleus /= pmf_nucleus.sum()
    return pmf_nucleus, cutoff

p_threshold = 0.9
pmf_nucleus, n_tokens_kept = nucleus_pmf(pmf, p_threshold)

print(f"Top-p = {p_threshold}: keeping {n_tokens_kept} tokens out of {V}")

cumulative_sorted = np.cumsum(pmf[order])
plt.plot(range(V), cumulative_sorted, marker="o")
plt.axhline(p_threshold, linestyle="--", label=f"p = {p_threshold}")
plt.axvline(n_tokens_kept - 1, linestyle=":", label=f"Nucleus cutoff ({n_tokens_kept} tokens)")
plt.xticks(range(V), [vocab[i] for i in order], rotation=45, ha="right")
plt.ylabel("Cumulative probability F(k)")
plt.title("Nucleus sampling uses the CDF to find the cutoff")
plt.legend()
plt.tight_layout()
plt.show()

## Step 5: Entropy — $E[-\log P(X)]$ measures model uncertainty

$$
H(X) = -\sum_k P(X = k) \log P(X = k)
$$

Entropy is the **expected negative log probability** — it is $E[X]$ where $X = -\log P(\text{token})$.

- Low entropy → model is confident (peaked PMF)
- High entropy → model is uncertain (flat PMF)
- Maximum entropy = $\log V$ (uniform distribution over vocabulary)

In [ ]:
def entropy(pmf):
    pmf = pmf[pmf > 0]
    return -np.sum(pmf * np.log(pmf))

for T in [0.3, 0.5, 1.0, 2.0, 5.0]:
    p = softmax(logits, T=T)
    H = entropy(p)
    max_H = np.log(V)
    print(f"T = {T:.1f}  →  H(X) = {H:.3f}  (max possible = {max_H:.3f})")

## Observation

- The **softmax output is a PMF** — every LLM call produces one, over a vocabulary of 50k–100k tokens.
- **Temperature** is a direct parameter of the PMF: it controls how sharply probability is concentrated on the top tokens.
- **Top-$k$ sampling** truncates the PMF to the $k$ most likely tokens.
- **Nucleus sampling** uses the **CDF** to find an adaptive cutoff: the nucleus is the smallest set of tokens whose cumulative probability reaches $p$.
- **Entropy** is $E[-\log P(X)]$ — a direct application of expected value to a function of the random variable. It is used in training (cross-entropy loss), in monitoring (detecting distributional shift), and in research (measuring model confidence).